In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('D:\sprints project\selected_data.csv')
df.head()

,PC4,PC5,PC2,PC1,PC8,PC3,PC9,PC7,PC6,PC10,Target
0,2.293052,0.023175,-1.087655,1.130664,-0.536787,3.164263,-1.495392,0.664854,0.578814,-0.499485,0
1,-0.857970,-0.006289,-1.417885,3.190926,1.069777,-0.533715,0.342524,-0.259063,0.745347,1.431509,1
2,-0.626641,0.152793,0.657008,3.124339,0.209299,-0.285134,0.043205,-0.324995,1.130179,0.462304,1
3,2.832741,0.721309,1.410972,-0.484339,-2.153525,0.397806,0.760079,-0.522221,-0.388361,0.228379,0
4,1.209318,0.770835,-0.330033,-2.284542,0.014736,-0.072260,1.050381,0.379567,0.625587,0.628110,0


### 1. Set up the baseline model

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split data
X = df.drop(columns=['Target'])
y = df['Target']

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,stratify=y)

# Baseline model
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

# Evaluate
y_pred = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred)
print(f"Baseline Accuracy: {baseline_acc:.4f}")


Baseline Accuracy: 0.8800


### 2. Hyperparameter tuning with GridSearchCV

In [6]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print(f"Best params (GridSearchCV): {grid_search.best_params_}")
best_grid_model = grid_search.best_estimator_

# Evaluate
y_pred_grid = best_grid_model.predict(X_test)
grid_acc = accuracy_score(y_test, y_pred_grid)
print(f"GridSearchCV Accuracy: {grid_acc:.4f}")

Best params (GridSearchCV): {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
GridSearchCV Accuracy: 0.8800


### 3. Hyperparameter tuning with RandomizedSearchCV

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Define parameter distribution
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5)
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,  
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)
print(f"Best params (RandomizedSearchCV): {random_search.best_params_}")
best_random_model = random_search.best_estimator_

# Evaluate
y_pred_random = best_random_model.predict(X_test)
random_acc = accuracy_score(y_test, y_pred_random)
print(f"RandomizedSearchCV Accuracy: {random_acc:.4f}")


Best params (RandomizedSearchCV): {'max_depth': 20, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 149}
RandomizedSearchCV Accuracy: 0.8800


### 4. Compare performances

In [8]:
print("Model Performance Comparison:")
print(f"Baseline Accuracy:        {baseline_acc:.4f}")
print(f"GridSearchCV Accuracy:    {grid_acc:.4f}")
print(f"RandomizedSearchCV Accuracy: {random_acc:.4f}")

Model Performance Comparison:
Baseline Accuracy:        0.8800
GridSearchCV Accuracy:    0.8800
RandomizedSearchCV Accuracy: 0.8800


In [9]:
import joblib

# Save the best RandomizedSearchCV model
joblib.dump(best_random_model, "model.pkl")

['model.pkl']

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Example: scaling + model
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Replace or add other preprocessing
    ('model', best_random_model)
])

# Fit the pipeline on training data
pipeline.fit(X_train, y_train)

# Save the pipeline
joblib.dump(pipeline, "final_model.pkl")

['final_model.pkl']